# IU model paylaşımı — yazılım doğrulaması (düzeltilmiş defter, v2)

**Aynı model ZIP'i:** IU_Open_Release_0_1_0rc1_Yayin_Adayi.zip.
Sonuç ZIP'ini bu deftere yüklemeyin.

Colab'da **L4, A100 veya H100** GPU seçin; yüksek RAM gerekmez.
Bu defter, Colab'ın mevcut CUDA/PyTorch kurulumunu kullanır; diğer paketleri
ayrı bir Python ortamında yükler. Model kodu da bu ortamın yeni Python
süreçlerinde çalışır. Colab hücresindeki paket önbelleğine dayanmaz.

Hücreleri sırayla çalıştırın. Hata alırsanız **5. hücreyi** yine çalıştırın.
Hata izi, aşama, kurulum çıktısı ve ortam bilgileri sonuç ZIP'ine kaydedilir.
Yeniden eğitim yapılmaz. Altı kurgusal görüntüyle yazılım karşılaştırması
yapılır; bu bir klinik başarı deneyi değildir.

In [ ]:
# 1 — Aynı yayın adayı ZIP'ini yükleyin.
from pathlib import Path
import os, sys, json, tempfile, zipfile, subprocess, shutil, traceback, time
from google.colab import files
WORK = Path(tempfile.mkdtemp(prefix="iu_release_v2_", dir=os.environ.get("IU_WORKSPACE_PARENT", "/content")))
RESULT_DIR = WORK / "validation_output"
RESULT_DIR.mkdir()
CURRENT_STAGE = "archive_upload"
def read_status():
    path = RESULT_DIR / "status.json"
    try:
        return json.loads(path.read_text())
    except (FileNotFoundError, json.JSONDecodeError):
        return {}
def update_status(**changes):
    data = read_status()
    data.update(notebook_setup_revision=2, software_validation_only=True, new_training_updates=0)
    data.update(changes)
    temporary = RESULT_DIR / "status.json.tmp"
    temporary.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    temporary.replace(RESULT_DIR / "status.json")
def capture_exception(error):
    trace = "".join(traceback.format_exception(type(error), error, error.__traceback__))
    (RESULT_DIR / "notebook_error.txt").write_text(trace, encoding="utf-8")
    update_status(status="FAILED", stage=CURRENT_STAGE,
                  notebook_error_type=type(error).__name__, notebook_error=str(error))
# Herhangi bir hücredeki beklenmeyen hatayı da kaydet.
ip = get_ipython()
if "_iu_error_hook" in globals():
    try:
        ip.events.unregister("post_run_cell", _iu_error_hook)
    except (ValueError, KeyError):
        pass
def _iu_error_hook(result):
    error = getattr(result, "error_before_exec", None) or getattr(result, "error_in_exec", None)
    if error is not None:
        capture_exception(error)
ip.events.register("post_run_cell", _iu_error_hook)
update_status(status="SETUP_PENDING", stage=CURRENT_STAGE)
uploaded = files.upload()
names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(names) != 1:
    raise ValueError("Yalnız IU_Open_Release yayın adayı ZIP'ini seçin.")
archive_path = WORK / Path(names[0]).name
archive_path.write_bytes(uploaded[names[0]])
del uploaded
with zipfile.ZipFile(archive_path) as archive:
    for entry in archive.infolist():
        target = (WORK / entry.filename).resolve()
        if not target.is_relative_to(WORK.resolve()) or ((entry.external_attr >> 16) & 0o170000) == 0o120000:
            raise ValueError("Geçersiz arşiv yolu.")
    archive.extractall(WORK / "package")
code_dirs = [p.parent for p in (WORK / "package").rglob("pyproject.toml") if p.parent.name == "github"]
if len(code_dirs) != 1:
    raise ValueError("Bu ZIP'te beklenen github klasörü yok. Yayın adayı ZIP'ini seçtiğinizden emin olun.")
CODE_DIR = code_dirs[0]
MODEL_DIR = CODE_DIR.parent / "huggingface_model"
assert (MODEL_DIR / "model_registry.json").is_file()
RUN_ENV = dict(os.environ)
RUN_ENV["PYTHONPATH"] = str(CODE_DIR / "src")
RUN_ENV["PYTHONUNBUFFERED"] = "1"
def run_logged(stage, command, log_name, env=None):
    update_status(status="RUNNING", stage=stage)
    log_path = RESULT_DIR / log_name
    print("Aşama:", stage, flush=True)
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            [str(item) for item in command], stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding="utf-8",
            errors="replace", env=RUN_ENV if env is None else env, bufsize=1)
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line.rstrip()[:1200], flush=True)
            return_code = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
            raise
    stages = read_status().get("completed_processes", [])
    stages.append({"stage": stage, "return_code": return_code, "log_file": log_name})
    update_status(completed_processes=stages)
    if return_code:
        update_status(status="FAILED", stage=stage, return_code=return_code, log_file=log_name)
        raise RuntimeError("Aşama tamamlanmadı: " + stage + ". Ayrıntılar " + log_name +
                           " dosyasında. 5. hücreyle sonuç ZIP'ini indirin.")
update_status(status="ARCHIVE_READY", stage="archive_ready")
print("ZIP hazır. Şimdi 2. hücreyi çalıştırın.")

In [ ]:
# 2 — Colab'ın ana ortamını değiştirmeden ayrı Python ortamı kurun.
CURRENT_STAGE = "isolated_environment_install"
import venv
VENV_DIR = WORK / "runtime"
venv.EnvBuilder(with_pip=False, system_site_packages=True).create(VENV_DIR)
WORKER_PYTHON = VENV_DIR / "bin" / "python"
# system_site_packages, Colab'ın mevcut CUDA/PyTorch paketlerini kullanır.
# pip --require-virtualenv, kurulumun bu ayrı ortamda yapılmasını zorunlu tutar.
run_logged("isolated_install", [
    WORKER_PYTHON, "-m", "pip", "--require-virtualenv", "install",
    "--disable-pip-version-check", "-e", CODE_DIR
], "installation.log")
update_status(status="INSTALLATION_COMPLETE", stage=CURRENT_STAGE)
print("Kurulum tamamlandı. Gradio çakışma uyarısı görülürse sonucu 3. hücredeki gerçek import kontrolü belirler.")

In [ ]:
# 3 — Yeni Python sürecinde paketleri, üç adaptörü ve GPU'yu kontrol edin.
CURRENT_STAGE = "environment_and_weights"
probe = '''
import sys, json, platform, traceback
from pathlib import Path
from importlib.metadata import version
output = Path(sys.argv[1])
model_dir = Path(sys.argv[2])
data = {"python": platform.python_version(), "executable": sys.executable}
try:
    import torch, numpy, diffusers, transformers, peft
    from diffusers import UNet2DConditionModel, AutoencoderKL, DDIMScheduler
    from transformers import CLIPTextModel, CLIPTokenizer
    from iu_cxr.integrity import verify_model
    data["packages"] = {name: version(name) for name in [
        "torch", "numpy", "diffusers", "transformers", "peft", "accelerate",
        "huggingface-hub", "safetensors", "Pillow"
    ]}
    data["torch_cuda_build"] = torch.version.cuda
    data["cuda_available"] = torch.cuda.is_available()
    data["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    data["bf16_supported"] = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
    checks = [verify_model(model_dir, seed) for seed in [42, 1337, 2026]]
    (output / "initial_checkpoint_checks.json").write_text(json.dumps(checks, indent=2))
    data["imports"] = "PASSED"
    if not data["cuda_available"] or not data["bf16_supported"]:
        raise RuntimeError("BF16 destekli L4/A100/H100 gibi bir Colab GPU seçin.")
    data["status"] = "PASSED"
except Exception as error:
    data.update(status="FAILED", error_type=type(error).__name__, error=str(error))
    (output / "preflight_error.txt").write_text(traceback.format_exc())
    raise
finally:
    (output / "initial_environment.json").write_text(json.dumps(data, indent=2))
    print(json.dumps(data, indent=2), flush=True)
'''
import textwrap
probe_path = WORK / "probe.py"
probe_path.write_text(textwrap.dedent(probe), encoding="utf-8")
run_logged("preflight", [WORKER_PYTHON, "-u", probe_path, RESULT_DIR, MODEL_DIR], "preflight.log")
update_status(status="PREFLIGHT_PASSED", stage=CURRENT_STAGE)
print("Paketler, adaptörler ve GPU kontrolü geçti. Şimdi 4. hücreyi çalıştırın.")

In [ ]:
# 4 — Altı kurgusal görüntüyle yazılım karşılaştırmasını çalıştırın.
CURRENT_STAGE = "GPU_software_validation"
if read_status().get("status") != "PREFLIGHT_PASSED":
    raise RuntimeError("Önce 3. hücrenin başarıyla tamamlanması gerekiyor.")
run_logged("generation", [
    WORKER_PYTHON, "-u", "-m", "iu_cxr.validation",
    "--model-dir", MODEL_DIR,
    "--examples-dir", CODE_DIR / "examples",
    "--output-dir", RESULT_DIR
], "generation.log")
status = read_status()
if status.get("status") != "PASSED":
    raise RuntimeError("Üretim kontrolü PASSED sonucu vermedi. 5. hücreyi çalıştırın.")
print("Yazılım doğrulaması geçti. Şimdi 5. hücreyle sonuçları indirin.")

In [ ]:
# 5 — Hata olsa da çalıştırın; sonuç ZIP'ini indirir.
if "RESULT_DIR" not in globals():
    raise RuntimeError("Önce 1. hücreyi çalıştırın.")
if not (RESULT_DIR / "status.json").exists():
    update_status(status="INCOMPLETE", stage=globals().get("CURRENT_STAGE", "unknown"))
result_zip = shutil.make_archive(str(WORK / "IU_Release_Dogrulama_Sonucu"), "zip", RESULT_DIR)
files.download(result_zip)
print("İndirilen IU_Release_Dogrulama_Sonucu.zip dosyasını paylaşın.")
print("Son durum:", read_status().get("status"))

**Sınır:** PASSED, paketlenmiş üretimin özgün kodla aynı ortamda karşılaştırma
kontrollerini geçtiğini gösterir. Klinik doğruluğu veya gizlilik garantisini
kanıtlamaz. Defterin v2 düzenlemesi yalnız çalıştırma ortamını ve hata
kaydını değiştirir; adaptör ağırlıkları ve üretim algoritması aynı kalır.